# STTran · 300 videos · Action Genome (zipped annotations + frames)

This notebook **clones your repo on Colab**, downloads **GloVe + Faster R-CNN + STTran weights**, merges **`annotations.zip`** and **`frames.zip`** into a valid AG tree, then runs **`run_first5_videos_all_frames.py`** with **`VIDEO_LIMIT=300`**.

**Outputs** (under `STTran/output/`):
- `logs/first5_videos/<VIDEO>.mp4.log` — full terminal-style log (parse for graphs / stats).
- `first5_videos/<VIDEO>.mp4/` — per-frame PNGs, GIF, legends.

After the run, **`export_graphs_json.py`** writes **`graphs.json`** in each video folder for programmatic graph extraction.

Push this repo (including `colab_minimal/`) to **GIT_REPO**, then edit the **config cell** paths (`GIT_REPO`, Drive zip paths). Use a **GPU** runtime before running.


In [ ]:
# --- EDIT THESE ---
GIT_REPO = "https://github.com/TommasoAiello08/STTran.git"
GIT_BRANCH = "main"  # change if your default branch differs

# Two zips on Google Drive (paths must exist after mounting Drive):
ANNOTATIONS_ZIP = "/content/drive/MyDrive/AG_zips/annotations.zip"
FRAMES_ZIP = "/content/drive/MyDrive/AG_zips/frames.zip"

# Where to build the Action Genome tree (ephemeral local disk on Colab):
AG_ROOT = "/content/ag_data"

# STTran code + outputs 
REPO_ROOT = "/content/STTran"

# How many videos to process (first N in frame_list order that appear in the chosen split)
VIDEO_LIMIT = 300
SPLIT = "test"   # "train" or "test"

# Optional: zip results back to Drive when done
OUTPUT_ZIP_ON_DRIVE = "/content/drive/MyDrive/sttran_outputs_300.zip"


In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

def run(cmd, cwd=None):
    print("+", " ".join(cmd), flush=True)
    subprocess.check_call(cmd, cwd=cwd)


def _rm_repo():
    p = Path(REPO_ROOT)
    if p.exists():
        shutil.rmtree(p, ignore_errors=True)
    subprocess.run(["rm", "-rf", REPO_ROOT], capture_output=True)


_rm_repo()

# Shallow clone on GIT_BRANCH. If that fails, nuke /content/STTran and clone default
# branch (avoids exit 128 when a failed first attempt left a non-empty path).
clone_branch = ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO, REPO_ROOT]
print("+", " ".join(clone_branch), flush=True)
r = subprocess.run(clone_branch, capture_output=True, text=True)
if r.returncode != 0:
    print("Branch clone failed:", r.stderr or r.stdout or "(no stderr)")
    _rm_repo()
    clone_default = ["git", "clone", "--depth", "1", GIT_REPO, REPO_ROOT]
    print("+", " ".join(clone_default), flush=True)
    r2 = subprocess.run(clone_default, capture_output=True, text=True)
    if r2.returncode != 0:
        print("Default-branch clone failed:", r2.stderr or r2.stdout or "(no stderr)")
        raise subprocess.CalledProcessError(r2.returncode, clone_default)

print(subprocess.check_output(["git", "log", "-1", "--oneline"], cwd=REPO_ROOT).decode())
os.chdir(REPO_ROOT)
print("REPO_ROOT:", REPO_ROOT)


In [ ]:
# Merge annotations.zip + frames.zip -> AG_ROOT/{annotations,frames}
from pathlib import Path
import importlib.util

ann = Path(ANNOTATIONS_ZIP)
frm = Path(FRAMES_ZIP)
assert ann.is_file(), f"Missing {ann}"
assert frm.is_file(), f"Missing {frm}"

spec = importlib.util.spec_from_file_location(
    "prepare_ag_zips",
    Path(REPO_ROOT) / "colab_minimal" / "prepare_ag_zips.py",
)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

mod.prepare_action_genome(ann, frm, Path(AG_ROOT), clean=True)
print("AG_ROOT ready:", AG_ROOT)


In [ ]:
# Install deps, compile extensions, GloVe, download official STTran + Faster R-CNN weights
# --colab = keep Colab's preinstalled torch (avoids pip conflicts). Use GPU runtime.
run([sys.executable, "setup_colab.py", "--colab"], cwd=REPO_ROOT)


In [ ]:
import os
os.environ["AG_DATA_PATH"] = AG_ROOT
os.environ["STTRAN_CKPT"] = str(Path(REPO_ROOT) / "ckpts" / "sttran_predcls.tar")
os.environ["SPLIT"] = SPLIT
os.environ["VIDEO_LIMIT"] = str(VIDEO_LIMIT)
os.environ["TOPK_PER_GROUP"] = "4"
os.environ["EDGE_THRESH"] = "0.0"
os.environ["VIZ_LAYOUT"] = "circular"
os.environ["VIZ_REUSE_LAYOUT"] = "1"
os.environ["MAX_RELS"] = "2000"

for k in ("AG_DATA_PATH", "VIDEO_LIMIT", "SPLIT"):
    print(k, os.environ.get(k))


In [ ]:
# Main run (~long): one folder per video under output/first5_videos/<VIDEO>.mp4/
run([sys.executable, "-u", "run_first5_videos_all_frames.py"], cwd=REPO_ROOT)


In [ ]:
# Parse logs -> graphs.json in each video folder (for downstream analysis)
import os
from pathlib import Path
os.environ.setdefault("TOPK_SPATIAL", "4")
os.environ.setdefault("TOPK_CONTACT", "4")
run([sys.executable, "export_graphs_json.py"], cwd=REPO_ROOT)


In [ ]:
# Optional: pack output/ into one zip on Drive (logs + viz + graphs.json)
from pathlib import Path
import shutil
out_dir = Path(REPO_ROOT) / "output"
if not out_dir.is_dir():
    print("No output directory — skipping zip.")
else:
    z = Path(OUTPUT_ZIP_ON_DRIVE)
    if z.suffix.lower() != ".zip":
        z = z.with_suffix(".zip")
    base = str(z.with_suffix(""))
    shutil.make_archive(base, "zip", root_dir=str(Path(REPO_ROOT)), base_dir="output")
    print("Wrote:", z)
